# PoliMillionaire - BM25 retrieval, no LLM

Runs all competitions through the API using the local SimpleWiki BM25 index. Each competition is played 5 times and every question attempt is printed and saved to CSV.

In [12]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "api_client" / "NLP_assignment_api_client").exists() and (path / "project" / "src").exists():
            return path
    raise FileNotFoundError("Could not find project root")

PROJECT_ROOT = find_project_root()
CLIENT_PARENT = PROJECT_ROOT / "api_client" / "NLP_assignment_api_client"
SRC_DIR = PROJECT_ROOT / "project" / "src"

for path in [str(CLIENT_PARENT), str(SRC_DIR)]:
    if path not in sys.path:
        sys.path.insert(0, path)

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\Tommaso\Code\NLP_polimi_26


In [13]:
from getpass import getpass
from millionaire_client import MillionaireClient
from retrieval_quiz_runner import load_retrieval_index, run_all_competitions, summarize, write_logs

API_URL = os.getenv("POLIMILLIONAIRE_API_URL", "http://131.175.15.22:51111/")
USERNAME = os.getenv("POLIMILLIONAIRE_USERNAME") or input("Username: ").strip()
PASSWORD = os.getenv("POLIMILLIONAIRE_PASSWORD") or getpass("Password: ")

client = MillionaireClient(API_URL, timeout=30)
user = client.login(USERNAME, PASSWORD)
print(f"Logged in as {user.username} ({user.role})")

Logged in as NeuroniNegroni (student)


In [14]:
INDEX_PATH = PROJECT_ROOT / "data" / "indexes" / "simplewiki_160w_bm25.joblib"
if not INDEX_PATH.exists():
    raise FileNotFoundError(f"Missing index: {INDEX_PATH}")

index = load_retrieval_index(INDEX_PATH)
print("Loaded index:", INDEX_PATH)
print("Kind:", index["kind"])
print("Chunks:", len(index["docs"]))

Loaded index: C:\Users\Tommaso\Code\NLP_polimi_26\data\indexes\simplewiki_160w_bm25.joblib
Kind: bm25
Chunks: 182397


In [15]:
competitions = client.competitions.list_all()
for comp in competitions:
    print(f"{comp.id}: {comp.name} | max levels: {comp.max_levels} | questions: {comp.question_count}")

0: Entertainment | max levels: 15 | questions: 843
1: Ancient History and Politics | max levels: 15 | questions: 456
2: Science and Nature | max levels: 15 | questions: 5395
3: Maths | max levels: 15 | questions: 390


In [16]:
ATTEMPTS_PER_COMPETITION = 5
STRATEGY_NAME = "simplewiki_bm25_no_llm"

rows = run_all_competitions(
    client=client,
    index=index,
    strategy_name=STRATEGY_NAME,
    attempts_per_competition=ATTEMPTS_PER_COMPETITION,
)

output_path = PROJECT_ROOT / "logs" / "simplewiki_bm25_no_llm_all_competitions.csv"
write_logs(rows, output_path)
print(f"Wrote {len(rows)} rows to {output_path}")

[simplewiki_bm25_no_llm] comp=0 Entertainment attempt=1 q=1 level=0 chosen=0 correct=False earned=0 latency=1.5693960000062361
Q: What is the fundamental principle of the phrase 'It's Over 9000!' in the context of the Dragon Ball Z episode?
A: It is used to describe the power of the Saiyan race.
Top evidence: Frieza :: Frieza (フリーザ, Furīza) is a fictional character in the Dragon Ball manga and the second major villain in the Dragon Ball Z anime.

Frieza is one of the strongest aliens in Dragon Ball Z. He enjoys invading other planets. Frieza is a very mean person who thinks he is very strong. He has destroyed planets, without mercy. He will even kill his own s...

[simplewiki_bm25_no_llm] comp=0 Entertainment attempt=2 q=1 level=0 chosen=0 correct=False earned=0 latency=1.368027999997139
Q: What is the fundamental principle of the phrase 'It's Over 9000!' in the context of the Dragon Ball Z episode?
A: It is used to describe the power of the Saiyan race.
Top evidence: Frieza :: Frieza 

In [17]:
summarize(rows)


Summary
0 Entertainment: rows=6, sessions=5, correct=1, row_acc=0.167, max_seen_level=0, avg_latency=0.844s
1 Ancient History and Politics: rows=7, sessions=5, correct=2, row_acc=0.286, max_seen_level=0, avg_latency=0.734s
2 Science and Nature: rows=8, sessions=5, correct=3, row_acc=0.375, max_seen_level=0, avg_latency=0.470s
3 Maths: rows=7, sessions=5, correct=2, row_acc=0.286, max_seen_level=0, avg_latency=0.962s
